In [1]:
import pandas as pd
import numpy as np

# =========================
# 1. Load data
# =========================
orders = pd.read_csv(
    "sim_orders.csv",
    parse_dates=["order_time", "reservation_time", "created_at"]
)

promotions = pd.read_csv(
    "sim_promotions.csv",
    parse_dates=["start_time", "end_time", "created_at"]
)

exposures = pd.read_csv(
    "sim_promotion_exposures.csv",
    parse_dates=["exposed_at", "clicked_at"]
)

# =========================
# 2. Exposure / click / redeem metrics
# =========================
metrics = (
    exposures
    .groupby(["promotion_id", "target_segment"], as_index=False)
    .agg(
        exposed_users=("user_id", "nunique"),
        total_exposures=("exposure_id", "count"),
        clicked_users=("clicked", "sum"),
        redeemed_users=("redeemed", "sum")
    )
)

metrics["click_rate"] = metrics["clicked_users"] / metrics["exposed_users"]
metrics["redeem_rate"] = metrics["redeemed_users"] / metrics["exposed_users"]

# =========================
# 3. Orders after exposure
# =========================
activity_rows = []

for _, row in exposures.iterrows():
    user_id = row["user_id"]
    exposed_at = row["exposed_at"]
    window_end = exposed_at + pd.Timedelta(days=7)

    after_orders = orders[
        (orders["user_id"] == user_id) &
        (orders["order_status"] == "completed") &
        (orders["order_time"] >= exposed_at) &
        (orders["order_time"] <= window_end)
    ]

    activity_rows.append({
        "promotion_id": row["promotion_id"],
        "target_segment": row["target_segment"],
        "user_id": user_id,
        "ordered_after_exposure": int(len(after_orders) > 0)
    })

activity_df = pd.DataFrame(activity_rows)

activity_user_level = (
    activity_df
    .groupby(["promotion_id", "target_segment", "user_id"], as_index=False)
    .agg(
        ordered_after_exposure=("ordered_after_exposure", "max")
    )
)

activity_metrics = (
    activity_user_level
    .groupby(["promotion_id", "target_segment"], as_index=False)
    .agg(
        ordered_after_exposure_users=("ordered_after_exposure", "sum")
    )
)

metrics = metrics.merge(
    activity_metrics,
    on=["promotion_id", "target_segment"],
    how="left"
)

metrics["ordered_after_exposure_users"] = (
    metrics["ordered_after_exposure_users"].fillna(0).astype(int)
)

metrics["order_after_exposure_rate"] = (
    metrics["ordered_after_exposure_users"] / metrics["exposed_users"]
)

# =========================
# 4. Add promotion information
# =========================
metrics = metrics.merge(
    promotions[
        [
            "promotion_id",
            "promotion_name",
            "restaurant_id",
            "cuisine",
            "discount_type",
            "discount_value",
            "start_time",
            "end_time"
        ]
    ],
    on="promotion_id",
    how="left"
)

metrics = metrics[
    [
        "promotion_id",
        "promotion_name",
        "restaurant_id",
        "cuisine",
        "target_segment",
        "discount_type",
        "discount_value",
        "start_time",
        "end_time",
        "exposed_users",
        "total_exposures",
        "clicked_users",
        "click_rate",
        "redeemed_users",
        "redeem_rate",
        "ordered_after_exposure_users",
        "order_after_exposure_rate"
    ]
]

# =========================
# 5. Monthly retention metrics
# =========================
# March exposure -> April retention
# April exposure -> May retention
# May exposure excluded

exposures["exposure_month"] = exposures["exposed_at"].dt.to_period("M")

retention_rows = []

for _, row in exposures.iterrows():
    exposure_month = row["exposure_month"]

    if exposure_month == pd.Period("2026-03", freq="M"):
        retention_month = "2026-04"
        retention_start = pd.Timestamp("2026-04-01")
        retention_end = pd.Timestamp("2026-04-30 23:59:59")

    elif exposure_month == pd.Period("2026-04", freq="M"):
        retention_month = "2026-05"
        retention_start = pd.Timestamp("2026-05-01")
        retention_end = pd.Timestamp("2026-05-31 23:59:59")

    else:
        continue

    user_id = row["user_id"]

    next_month_orders = orders[
        (orders["user_id"] == user_id) &
        (orders["order_status"] == "completed") &
        (orders["order_time"] >= retention_start) &
        (orders["order_time"] <= retention_end)
    ]

    retention_rows.append({
        "promotion_id": row["promotion_id"],
        "target_segment": row["target_segment"],
        "user_id": user_id,
        "retention_month": retention_month,
        "retained_next_month": int(len(next_month_orders) > 0)
    })

retention_df = pd.DataFrame(retention_rows)

retention_user_level = (
    retention_df
    .groupby(
        ["promotion_id", "target_segment", "user_id", "retention_month"],
        as_index=False
    )
    .agg(
        retained_next_month=("retained_next_month", "max")
    )
)

retention_metrics = (
    retention_user_level
    .groupby(["promotion_id", "target_segment", "retention_month"], as_index=False)
    .agg(
        retention_evaluable_users=("user_id", "nunique"),
        retained_next_month_users=("retained_next_month", "sum")
    )
)

retention_metrics["next_month_retention_rate"] = (
    retention_metrics["retained_next_month_users"]
    / retention_metrics["retention_evaluable_users"]
)

retention_metrics = retention_metrics.merge(
    promotions[
        [
            "promotion_id",
            "promotion_name",
            "restaurant_id",
            "cuisine",
            "discount_type",
            "discount_value"
        ]
    ],
    on="promotion_id",
    how="left"
)

retention_metrics = retention_metrics[
    [
        "promotion_id",
        "promotion_name",
        "restaurant_id",
        "cuisine",
        "target_segment",
        "discount_type",
        "discount_value",
        "retention_month",
        "retention_evaluable_users",
        "retained_next_month_users",
        "next_month_retention_rate"
    ]
]

# =========================
# 6. Export
# =========================
metrics.to_csv("sim_promotion_metrics.csv", index=False)

retention_metrics.to_csv(
    "sim_monthly_retention_metrics.csv",
    index=False
)

display(metrics.head(20))
display(retention_metrics.head(20))

,promotion_id,promotion_name,restaurant_id,cuisine,target_segment,discount_type,discount_value,start_time,end_time,exposed_users,total_exposures,clicked_users,click_rate,redeemed_users,redeem_rate,ordered_after_exposure_users,order_after_exposure_rate
0,1,Coffee/Tea Promo 1,50150310,Coffee/Tea,dormant_users,percentage,0.25,2026-05-05,2026-05-12,141,141,20,0.141844,0,0.0,27,0.191489
1,2,American Promo 2,50001835,American,all_users,fixed,12.00,2026-03-14,2026-03-17,450,450,72,0.160000,0,0.0,120,0.266667
2,4,Korean Promo 4,50135792,Korean,all_users,percentage,0.25,2026-04-21,2026-04-24,450,450,57,0.126667,0,0.0,116,0.257778
3,5,Chinese Promo 5,50146135,Chinese,all_users,percentage,0.25,2026-04-29,2026-05-04,450,450,86,0.191111,0,0.0,124,0.275556
4,6,Latin American Promo 6,40391272,Latin American,all_users,percentage,0.10,2026-04-28,2026-05-05,450,450,89,0.197778,0,0.0,118,0.262222
5,7,Donuts Promo 7,50094935,Donuts,dormant_users,fixed,10.00,2026-05-02,2026-05-16,136,136,20,0.147059,0,0.0,22,0.161765
6,8,Japanese Promo 8,50074873,Japanese,all_users,percentage,0.20,2026-04-23,2026-04-30,450,450,92,0.204444,0,0.0,110,0.244444
7,9,Sandwiches Promo 9,50044606,Sandwiches,dormant_users,percentage,0.10,2026-05-10,2026-05-15,144,144,24,0.166667,0,0.0,23,0.159722
8,10,Other Promo 10,50181305,Other,all_users,percentage,0.10,2026-04-20,2026-04-25,450,450,74,0.164444,0,0.0,123,0.273333
9,11,"Juice, Smoothies, Fruit Salads Promo 11",50063197,"Juice, Smoothies, Fruit Salads",new_users,percentage,0.20,2026-03-05,2026-03-12,516,516,119,0.230620,0,0.0,128,0.248062


,promotion_id,promotion_name,restaurant_id,cuisine,target_segment,discount_type,discount_value,retention_month,retention_evaluable_users,retained_next_month_users,next_month_retention_rate
0,2,American Promo 2,50001835,American,all_users,fixed,12.00,2026-04,450,291,0.646667
1,4,Korean Promo 4,50135792,Korean,all_users,percentage,0.25,2026-05,450,299,0.664444
2,5,Chinese Promo 5,50146135,Chinese,all_users,percentage,0.25,2026-05,152,109,0.717105
3,6,Latin American Promo 6,40391272,Latin American,all_users,percentage,0.10,2026-05,166,119,0.716867
4,8,Japanese Promo 8,50074873,Japanese,all_users,percentage,0.20,2026-05,450,304,0.675556
5,10,Other Promo 10,50181305,Other,all_users,percentage,0.10,2026-05,450,293,0.651111
6,11,"Juice, Smoothies, Fruit Salads Promo 11",50063197,"Juice, Smoothies, Fruit Salads",new_users,percentage,0.20,2026-04,516,328,0.635659
7,12,Japanese Promo 12,41628657,Japanese,all_users,fixed,10.00,2026-04,450,288,0.640000
8,14,Italian Promo 14,50047472,Italian,new_users,fixed,10.00,2026-04,211,120,0.568720
9,14,Italian Promo 14,50047472,Italian,new_users,fixed,10.00,2026-05,45,25,0.555556


In [3]:
import pandas as pd
import numpy as np

# =========================
# 1. Load data
# =========================
users = pd.read_csv("sim_users.csv")
restaurants_wait = pd.read_csv("restaurant_wait_now.csv")
restaurants_clean = pd.read_csv("restaurant_clean.csv")

promotions = pd.read_csv(
    "sim_promotions.csv",
    parse_dates=["start_time", "end_time"]
)

# add lat/lon to restaurant_wait_now
restaurants = restaurants_wait.merge(
    restaurants_clean[["restaurant_id", "lat", "lon", "inspection_grade", "inspection_score"]],
    on="restaurant_id",
    how="left"
)

restaurants = restaurants.dropna(subset=["lat", "lon"])

# =========================
# 2. Utility functions
# =========================

def haversine_minutes(lat1, lon1, lat2, lon2, speed_kmh=18):
    """
    Approximate urban travel time.
    speed_kmh=18 means mixed walking/transit/short taxi average.
    """
    R = 6371
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    distance_km = 2 * R * np.arcsin(np.sqrt(a))
    minutes = distance_km / speed_kmh * 60

    return minutes


def minmax_normalize(series):
    if series.max() == series.min():
        return pd.Series(0.5, index=series.index)
    return (series - series.min()) / (series.max() - series.min())


def preference_score(row, user):
    if row["cuisine"] == user["preferred_cuisine_1"]:
        return 1.0
    elif row["cuisine"] == user["preferred_cuisine_2"]:
        return 0.7
    else:
        return 0.2


def get_active_promotion_score(restaurants_df, current_time):
    active_promos = promotions[
        (promotions["start_time"] <= current_time) &
        (promotions["end_time"] >= current_time)
    ].copy()

    if len(active_promos) == 0:
        restaurants_df["promotion_bonus"] = 0.0
        restaurants_df["active_promotion_id"] = np.nan
        return restaurants_df

    active_promos["promotion_bonus"] = np.where(
        active_promos["discount_type"] == "percentage",
        active_promos["discount_value"],
        active_promos["discount_value"] / 50
    )

    promo_score = (
        active_promos
        .groupby("restaurant_id", as_index=False)
        .agg(
            promotion_bonus=("promotion_bonus", "max"),
            active_promotion_id=("promotion_id", "first")
        )
    )

    restaurants_df = restaurants_df.merge(
        promo_score,
        on="restaurant_id",
        how="left"
    )

    restaurants_df["promotion_bonus"] = restaurants_df["promotion_bonus"].fillna(0)
    return restaurants_df


# =========================
# 3. Recommendation function
# =========================

def recommend_restaurants(
    user_id,
    origin_lat,
    origin_lng,
    current_time="2026-05-15 18:30:00",
    top_n=5,
    max_travel_minutes=45,
    max_wait_minutes=60,
    weight_time=0.50,
    weight_utilization=0.25,
    weight_preference=0.20,
    weight_promotion=0.05
):
    current_time = pd.Timestamp(current_time)

    user = users.loc[users["user_id"] == user_id].iloc[0]
    df = restaurants.copy()

    # travel time
    df["estimated_travel_minutes"] = haversine_minutes(
        origin_lat,
        origin_lng,
        df["lat"],
        df["lon"]
    )

    # user total time
    df["estimated_wait_minutes"] = df["estimated_wait_minutes"].fillna(0)
    df["total_user_time"] = (
        df["estimated_travel_minutes"] + df["estimated_wait_minutes"]
    )

    # current utilization
    df["current_utilization"] = (
        df["effective_demand"]
        / df["effective_supply_index"].clip(lower=1)
    )

    # idle capacity: higher means restaurant has more idle capacity
    df["idle_capacity_score"] = (1 - df["current_utilization"]).clip(lower=0, upper=1)

    # preference
    df["preference_match_score"] = df.apply(
        lambda row: preference_score(row, user),
        axis=1
    )

    # active promotion
    df = get_active_promotion_score(df, current_time)

    # filter infeasible options
    df = df[
        (df["estimated_travel_minutes"] <= max_travel_minutes) &
        (df["estimated_wait_minutes"] <= max_wait_minutes)
    ].copy()

    if len(df) == 0:
        return pd.DataFrame()

    # normalize time cost
    df["normalized_total_user_time"] = minmax_normalize(df["total_user_time"])

    # final score
    df["final_score"] = (
        - weight_time * df["normalized_total_user_time"]
        + weight_utilization * df["idle_capacity_score"]
        + weight_preference * df["preference_match_score"]
        + weight_promotion * df["promotion_bonus"]
    )

    df = df.sort_values("final_score", ascending=False).copy()
    df["rank"] = range(1, len(df) + 1)

    result = df.head(top_n)[
        [
            "rank",
            "restaurant_id",
            "name",
            "cuisine",
            "address",
            "estimated_travel_minutes",
            "estimated_wait_minutes",
            "total_user_time",
            "current_utilization",
            "idle_capacity_score",
            "preference_match_score",
            "promotion_bonus",
            "active_promotion_id",
            "final_score"
        ]
    ].copy()

    result.insert(0, "user_id", user_id)
    result.insert(1, "current_time", current_time)
    result.insert(2, "origin_lat", origin_lat)
    result.insert(3, "origin_lng", origin_lng)

    return result


# =========================
# 4. Generate batch recommendations
# =========================
# For simulation only: randomly generate current user origins around Manhattan.
# In real app, these should come from GPS or manual input.

np.random.seed(42)

manhattan_lat_min, manhattan_lat_max = 40.7000, 40.8800
manhattan_lng_min, manhattan_lng_max = -74.0200, -73.9300

recommendation_results = []

for user_id in users["user_id"]:
    origin_lat = np.random.uniform(manhattan_lat_min, manhattan_lat_max)
    origin_lng = np.random.uniform(manhattan_lng_min, manhattan_lng_max)

    recs = recommend_restaurants(
        user_id=user_id,
        origin_lat=origin_lat,
        origin_lng=origin_lng,
        current_time="2026-05-15 18:30:00",
        top_n=5
    )

    recommendation_results.append(recs)

recommendation_results = pd.concat(recommendation_results, ignore_index=True)

# =========================
# 5. Export
# =========================

recommendation_results.to_csv(
    "sim_recommendation_results.csv",
    index=False
)

print("Recommendation results:", recommendation_results.shape)
display(recommendation_results.head(20))

Recommendation results: (5000, 18)


,user_id,current_time,origin_lat,origin_lng,rank,restaurant_id,name,cuisine,address,estimated_travel_minutes,estimated_wait_minutes,total_user_time,current_utilization,idle_capacity_score,preference_match_score,promotion_bonus,active_promotion_id,final_score
0,1,2026-05-15 18:30:00,40.767417,-73.934436,1,50176894,THAI HOT BOX,Thai,"1598 3 AVENUE, MANHATTAN, NY 10128",7.148958,0.0,7.148958,0.002907,0.997093,1.0,0.0,NaN,0.427870
1,1,2026-05-15 18:30:00,40.767417,-73.934436,2,50098846,THAI @ LEX,Thai,"1244 LEXINGTON AVENUE, MANHATTAN, NY 10028",7.375354,0.0,7.375354,0.003412,0.996588,1.0,0.0,NaN,0.426024
2,1,2026-05-15 18:30:00,40.767417,-73.934436,3,50072966,BANGKLYN EAST HARLEM,Thai,"2051 2 AVENUE, MANHATTAN, NY 10029",8.768967,0.0,8.768967,0.147168,0.852832,1.0,0.0,NaN,0.379501
3,1,2026-05-15 18:30:00,40.767417,-73.934436,4,50082311,KATI SHOP,Thai,"162 EAST 55 STREET, MANHATTAN, NY 10022",10.245878,0.0,10.245878,0.134681,0.865319,1.0,0.0,NaN,0.371407
4,1,2026-05-15 18:30:00,40.767417,-73.934436,5,50101251,THAI SUPER,Thai,"166 EAST 118 STREET, MANHATTAN, NY 10035",11.994895,0.0,11.994895,0.082547,0.917453,1.0,0.0,NaN,0.371157
5,2,2026-05-15 18:30:00,40.831759,-73.966121,1,50186471,PIZZA COFFEE SHOP MAMA VIOLA,Pizza,"3594 BROADWAY, MANHATTAN, NY 10031",4.935596,0.0,4.935596,0.031257,0.968743,1.0,0.0,NaN,0.438494
6,2,2026-05-15 18:30:00,40.831759,-73.966121,2,50005390,JUMBO PIZZA,Pizza,"3594 BROADWAY, MANHATTAN, NY 10031",4.935596,0.0,4.935596,0.031257,0.968743,1.0,0.0,NaN,0.438494
7,2,2026-05-15 18:30:00,40.831759,-73.966121,3,50105926,PIZZERIA L'ANTICA,Pizza,"3789 BROADWAY, MANHATTAN, NY 10032",6.089695,0.0,6.089695,0.017241,0.982759,1.0,0.0,NaN,0.434974
8,2,2026-05-15 18:30:00,40.831759,-73.966121,4,50134858,IPIZZANY,Pizza,"3801 BROADWAY, MANHATTAN, NY 10032",6.202470,0.0,6.202470,0.025444,0.974556,1.0,0.0,NaN,0.432237
9,2,2026-05-15 18:30:00,40.831759,-73.966121,5,41716092,DOMINO'S,Pizza,"3624 BROADWAY, MANHATTAN, NY 10031",5.050918,0.0,5.050918,0.061094,0.938906,1.0,0.0,NaN,0.430333
